# 🌲 Phase 2: Model Debugging & Feature Importance
**Project:** Machine Learning Experimental Design — Random Forest Churn Prediction

This notebook covers the single-model prototyping layer. We fit our `RandomForestClassifier` on the entire clean (multicollinearity-free) dataset, evaluate its F1-score, and analyze its **Gini Feature Importance**.

In [ ]:
import sys
from pathlib import Path

# Ensure project modules are importable
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, f1_score

from src.data_loader import load_data
from src.feature_analysis import (
    remove_highly_correlated_features,
    compute_feature_importance,
    plot_feature_importance
)
from src.model import build_rf

sns.set_theme(style="whitegrid")

## 1. Prepare Features & Target

In [ ]:
X, y = load_data()
X_clean, dropped = remove_highly_correlated_features(X, threshold=0.95)
print(f"Clean features shape: {X_clean.shape}")

## 2. Train Random Forest (Fully Grown)

In [ ]:
# build_rf uses our exact fixed seed 1234
model = build_rf(max_depth=None)
model.fit(X_clean, y)
print(model)

## 3. Quick Performance Check (Self-Fit / Overfitting diagnostic)

In [ ]:
y_pred = model.predict(X_clean)
print("Training Set Classification Report:")
print(classification_report(y, y_pred))

## 4. Extract and Plot Feature Importance

In [ ]:
importances = compute_feature_importance(model, X_clean.columns.tolist())
print("Feature Importances:")
print(importances)

In [ ]:
plot_feature_importance(importances, title="Random Forest Churn Feature Importances")
plt.show()

## 📌 Discussion of Findings

### Feature Importance Insights:
1. **`total_day_minutes`**: By far the most important feature (~26.7%). Customers with extremely high daily calling duration represent the highest risk of churn.
2. **`number_customer_service_calls`**: Second most important (~11.9%). Highly actionable for operations: multiple calls to customer service strongly indicate ongoing customer dissatisfaction.
3. **`total_eve_minutes`**: Third most important (~10.1%). Afternoon usage also has strong churn signals.
4. **`international_plan`**: Fourth most important (~8.8%). Subscribers with international calling configurations exhibit specialized churn behaviors.

### Training Performance Notes:
On self-fit (the entire training set), a fully-grown Random Forest (`max_depth=None`) yields **100% Accuracy / 1.0 F1-score**. This is classic overfitting due to tree leaf purity, highlighting the absolute necessity of our **Repeated Cross-Validation** (CRD and CRFD) pipelines to obtain realistic, generalizable generalization scores!